# Scales and ticks

**Scales and ticks -- the axis is an argument, not furniture.**

**What it shows:**

- a log scale turns "I can only see the biggest one" into a readable chart
- log is for ratios; it is the wrong tool for data containing zero
- tick formatting: thousands separators, percentages, currency
- date axes, which matplotlib handles specially

---

*Chapter:* `foundations` — matplotlib's actual mechanics  
*Run the cells in order.* Every figure is also written to `viz/output/foundations/`, which is what the Streamlit gallery (`viz/project/gallery.py`) reads.


## Setup

These lines are how every notebook in the folder finds `vizkit.py`, which holds the save helpers and the seeded sample data. The data is seeded on purpose: your figures should come out identical to everyone else's.

`save()` writes each figure into `viz/output/` **and** leaves it on screen here. The trailing `;` on those calls only stops the notebook echoing the path it returns.


In [ ]:
%matplotlib inline

# A notebook has no __file__, so find viz/ by walking up from this
# notebook's own folder until vizkit.py turns up.
import sys
from pathlib import Path

VIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
           if (p / "vizkit.py").exists())
sys.path.insert(0, str(VIZ))

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

from vizkit import save, temperatures

# Where save() files this lesson's output: viz/output/foundations/
LESSON = "foundations/scales_and_ticks"


## 1. When a linear axis hides everything

On the linear axis, five of the seven countries are a smear at the left edge. The log axis makes every value readable — at the price that equal distances now mean equal **ratios**, which you must say out loud in the label.


In [ ]:
countries = ["Tuvalu", "Iceland", "Ireland", "Poland", "Brazil", "USA", "China"]
gdp = [0.06, 30, 550, 810, 2170, 27360, 17790]      # billions USD

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))

left.barh(countries, gdp, color="#4C72B0")
left.set_title("Linear: five of the seven bars are invisible")
left.set_xlabel("GDP ($bn)")

right.barh(countries, gdp, color="#4C72B0")
right.set_xscale("log")
right.set_title("Log: every value readable, but read it as RATIOS")
right.set_xlabel("GDP ($bn), log scale")

fig.suptitle("Log scale: each step is a multiple, not an addition", fontsize=11)
fig.tight_layout()
save(fig, LESSON, "log-scale");


## 2. What a log scale costs you

This is the clearest thing a log scale does: exponential growth becomes a straight line, so a bend in that line is a change in the growth *rate*. The linear panel's 'it exploded' is an artefact.


In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))

x = np.arange(1, 25)
doubling = 2.0 ** (x / 3)

left.plot(x, doubling, "-o", color="#C44E52", markersize=3)
left.set_title("Linear: 'it exploded at the end'")

right.plot(x, doubling, "-o", color="#C44E52", markersize=3)
right.set_yscale("log")
right.set_title("Log: a straight line = a CONSTANT growth rate")

fig.suptitle("Same data. The log version reveals that nothing changed.",
             fontsize=11)
fig.tight_layout()
save(fig, LESSON, "log-reveals-rate");


## 3. Tick formatting

A formatter is just a function matplotlib calls for each tick value. Two lines of it saves every reader from counting zeros.


In [ ]:
revenue = np.array([1_250_000, 2_100_000, 3_400_000, 4_050_000, 5_600_000])
quarters = ["Q1", "Q2", "Q3", "Q4", "Q5"]

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))

left.bar(quarters, revenue, color="#4C72B0")
left.set_title("Default: what am I looking at? 1e6?")

right.bar(quarters, revenue, color="#4C72B0")
# A formatter is a function matplotlib calls for each tick value.
right.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"${v/1e6:.1f}M"))
right.set_title("Formatted: units on the ticks, not in a footnote")

fig.tight_layout()
save(fig, LESSON, "tick-format");


## 4. Date axes

Dates get their own machinery: a **Locator** decides where ticks go, a **Formatter** decides how each one reads. They are independent, which is why you can have monthly ticks labelled by year.


In [ ]:
import matplotlib.dates as mdates                    # noqa: E402

temps = temperatures()

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))

left.plot(temps["date"], temps["temp_c"], lw=0.8, color="#4C72B0")
left.set_title("Default date ticks")
left.tick_params(axis="x", rotation=45)

right.plot(temps["date"], temps["temp_c"], lw=0.8, color="#4C72B0")
right.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
right.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
right.set_title("Locator picks WHERE, formatter picks HOW it reads")

for ax in (left, right):
    ax.set_ylabel("temp (C)")

fig.tight_layout()
save(fig, LESSON, "date-axis");


## Rules of thumb

```text
The axis is part of the argument:
  huge range, all positive   -> log scale (ratios, never zero)
  straight line on a log y   -> constant growth rate
  big numbers                -> format the ticks, do not make people count zeros
  dates                      -> Locator = where, Formatter = how it reads
```


## Try it yourself

Edit the cells above and re-run them — that is what the notebook is for.

1. Add a country with GDP 0 to section 1 and re-run. What does the log panel do with it, and why is that the right complaint?
2. In section 2, change `2.0 ** (x / 3)` to `2.0 ** (x / 3) + 0.4 * x`. Can you still see the departure from constant growth on the log panel?
3. In section 4, use `mdates.MonthLocator(interval=1)` with `DateFormatter('%b %d')`. Fix the collision you just created without rotating the labels.


In [ ]:
# your turn


---

**Previous:** [`foundations/figure_and_axes`](figure_and_axes.ipynb)  
**Next:** [`foundations/subplots_grid`](subplots_grid.ipynb)
